# Clasificación de supervivencia en Titanic

| Integrante | Tareas realizadas |
|---|---|
| Integrante 1 | Limpieza de datos, feature engineering y pipeline |
| Integrante 2 | Entrenamiento de modelos y búsqueda de hiperparámetros |
| Integrante 3 | Evaluación, métricas, matriz de confusión y curva ROC |
| Integrante 4 | Análisis de overfitting, importancia de variables e informe |

> Reemplazar los nombres de los integrantes antes de entregar.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape)
print(test.shape)
train.head()


## Feature engineering

Se crean variables dentro del pipeline para evitar data leakage. Las variables nuevas incluyen título, tamaño familiar, pasajero solo, tarifa por persona, cabina registrada, cubierta, interacción edad-clase y tamaño del grupo de ticket.


In [ ]:
def add_features(X):
    X = X.copy()
    X['Title'] = X['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False).fillna('Unknown')
    X['Title'] = X['Title'].where(X['Title'].isin(['Mr', 'Miss', 'Mrs', 'Master']), 'Rare')
    X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
    X['IsAlone'] = (X['FamilySize'] == 1).astype(int)
    X['FarePerPerson'] = X['Fare'] / X['FamilySize']
    X['HasCabin'] = X['Cabin'].notna().astype(int)
    X['Deck'] = X['Cabin'].astype(str).str[0]
    X.loc[X['Cabin'].isna(), 'Deck'] = 'Unknown'
    X['AgeClass'] = X['Age'] * X['Pclass']
    X['TicketGroupSize'] = X.groupby('Ticket')['Ticket'].transform('count')
    return X

numeric_features = [
    'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize',
    'IsAlone', 'FarePerPerson', 'HasCabin', 'AgeClass', 'TicketGroupSize'
]

categorical_features = ['Sex', 'Embarked', 'Title', 'Deck']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = Pipeline(steps=[
    ('features', FunctionTransformer(add_features, validate=False)),
    ('columns', ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]))
])


## División de datos y modelos

La evaluación se realiza sobre una partición de `train.csv`, porque `test.csv` no incluye la variable `Survived`.


In [ ]:
X = train.drop(columns=['Survived'])
y = train['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}


In [ ]:
param_spaces = {
    'Decision Tree': {
        'model__max_depth': [3, 5, 7],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2]
    },
    'Random Forest': {
        'model__n_estimators': [80, 120],
        'model__max_depth': [5, None],
        'model__min_samples_leaf': [1, 2]
    },
    'Gradient Boosting': {
        'model__n_estimators': [50, 80],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [2, 3]
    }
}

best_models = {}
search_results = {}

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])

    if name == 'Decision Tree':
        search = GridSearchCV(pipe, param_spaces[name], cv=5, scoring='f1_macro', n_jobs=1)
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        search_results[name] = {
            'method': 'GridSearchCV',
            'best_params': search.best_params_,
            'best_score_f1_macro': search.best_score_
        }
    elif name in ['Random Forest', 'Gradient Boosting']:
        search = RandomizedSearchCV(
            pipe, param_spaces[name], n_iter=4, cv=5,
            scoring='f1_macro', random_state=42, n_jobs=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        search_results[name] = {
            'method': 'RandomizedSearchCV',
            'best_params': search.best_params_,
            'best_score_f1_macro': search.best_score_
        }
    else:
        pipe.fit(X_train, y_train)
        best_models[name] = pipe

search_results


In [ ]:
results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    train_f1 = f1_score(y_train, model.predict(X_train), average='macro')
    cv_f1 = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=1).mean()

    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision macro': precision_score(y_test, y_pred, average='macro'),
        'Recall macro': recall_score(y_test, y_pred, average='macro'),
        'F1 macro': f1_score(y_test, y_pred, average='macro'),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Train F1 macro': train_f1,
        'CV F1 macro': cv_f1,
        'Brecha Train-CV': train_f1 - cv_f1
    })

results_df = pd.DataFrame(results).sort_values('F1 macro', ascending=False)
results_df


In [ ]:
best_model_name = results_df.iloc[0]['Modelo']
best_model = best_models[best_model_name]

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

ConfusionMatrixDisplay(cm, display_labels=['No sobrevivió', 'Sobrevivió']).plot(values_format='d')
plt.title(f'Matriz de confusión - {best_model_name}')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))

for name, model in best_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    RocCurveDisplay.from_predictions(
        y_test, y_proba, name=f'{name} AUC={auc:.3f}', ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('Curvas ROC de los modelos')
plt.show()


In [ ]:
model_step = best_model.named_steps['model']
column_transformer = best_model.named_steps['preprocessor'].named_steps['columns']
feature_names = list(column_transformer.get_feature_names_out())

if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
else:
    importances = abs(model_step.coef_[0])

importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importancia': importances
}).sort_values('Importancia', ascending=False).head(10)

importance_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df['Variable'], importance_df['Importancia'])
plt.gca().invert_yaxis()
plt.title(f'Top 10 variables más importantes - {best_model_name}')
plt.xlabel('Importancia')
plt.show()


In [ ]:
# Entrenamiento final con todo train.csv y predicción para test.csv
final_model = best_model
final_model.fit(X, y)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_model.predict(test)
})

submission.to_csv('predicciones_test.csv', index=False)
submission.head()
